Problem 1: Let A be an array of activations of shape float32[SX, DY] with X * Y = N. Do the following:

Write a function in JAX that computes the average within each (X, Y) shard, i.e. it returns an array of size [X, Y] where arr[i, j] is the average over shard (i, j). Do this with both jax.jit and shard_map. Profile each and see how long they took. Was there any communication added? Hint: there shouldn’t be, but sometimes XLA adds it anyway.

Write a function in JAX that returns roll(x, shift, axis=0) - x for some shift within each shard X. I’m not enough of a masochist to make you do this in jax.jit, so just do this with shard_map.

In [1]:
# necessary imports
import os

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "0"        # verbose XLA logs
os.environ["JAX_TRACEBACK_FILTERING"] = "off"   # full Python traceback
## very very important for async all to all optimisation in MoE models
os.environ["LIBTPU_INIT_ARGS"] = "--xla_tpu_enable_async_all_to_all=true"
os.environ["LIBTPU_INIT_ARGS"] = "--xla_tpu_enable_async_collective_fusion=true"
import jax
import jax.numpy as jnp
import jax.sharding as shd
from jax.sharding import PartitionSpec as P
from jax.sharding import NamedSharding
import numpy as np
import math


# Control the backend to initialize only the CPU
# if jax.devices()[0].platform != 'tpu' and jax.devices()[0].platform != 'cpu':
# jax.config.update('jax_num_cpu_devices', 8)

print(f"Using {jax.device_count()} devices: {jax.devices()}")

/usr/local/lib/python3.12/site-packages/jax/_src/cloud_tpu_init.py:93: UserWarning: Transparent hugepages are not enabled. TPU runtime startup and shutdown time should be significantly improved on TPU v5e and newer. If not already set, you may need to enable transparent hugepages in your VM image (sudo sh -c "echo always > /sys/kernel/mm/transparent_hugepage/enabled")
  warnings.warn(


I0329 09:51:26.946152      74 pjrt_api.cc:118] GetPjrtApi was found for tpu at /usr/local/lib/python3.12/site-packages/libtpu/libtpu.so
I0329 09:51:26.946186      74 pjrt_api.cc:96] PJRT_Api is set for device type tpu
I0329 09:51:26.946692      74 pjrt_api.cc:167] The PJRT plugin has PJRT API version 0.69. The framework PJRT API version is 0.90.
E0000 00:00:1774777886.946868      74 common_lib.cc:648] Could not set metric server port: INVALID_ARGUMENT: Could not find SliceBuilder port 8471 in any of the 0 ports provided in `tpu_process_addresses`="local"
=== Source Location Trace: === 
learning/45eac/tfrc/runtime/common_lib.cc:238


Using 8 devices: [TpuDevice(id=0, process_index=0, coords=(0,0,0), core_on_chip=0), TpuDevice(id=1, process_index=0, coords=(1,0,0), core_on_chip=0), TpuDevice(id=2, process_index=0, coords=(0,1,0), core_on_chip=0), TpuDevice(id=3, process_index=0, coords=(1,1,0), core_on_chip=0), TpuDevice(id=4, process_index=0, coords=(0,2,0), core_on_chip=0), TpuDevice(id=5, process_index=0, coords=(1,2,0), core_on_chip=0), TpuDevice(id=6, process_index=0, coords=(0,3,0), core_on_chip=0), TpuDevice(id=7, process_index=0, coords=(1,3,0), core_on_chip=0)]


I0329 09:51:37.176873      74 pjrt_c_api_client.cc:168] PjRtCApiClient created.
I0329 09:51:37.176939      74 pjrt_client.cc:551] PjRt-IFRT device count: total=8, addressable=8
I0329 09:51:37.176947      74 pjrt_client.cc:555] Addressable PjRt-IFRT device: TpuDevice(id=0, process_index=0, coords=(0,0,0), core_on_chip=0)
I0329 09:51:37.176949      74 pjrt_client.cc:555] Addressable PjRt-IFRT device: TpuDevice(id=1, process_index=0, coords=(1,0,0), core_on_chip=0)
I0329 09:51:37.176951      74 pjrt_client.cc:555] Addressable PjRt-IFRT device: TpuDevice(id=2, process_index=0, coords=(0,1,0), core_on_chip=0)
I0329 09:51:37.176953      74 pjrt_client.cc:555] Addressable PjRt-IFRT device: TpuDevice(id=3, process_index=0, coords=(1,1,0), core_on_chip=0)
I0329 09:51:37.176955      74 pjrt_client.cc:555] Addressable PjRt-IFRT device: TpuDevice(id=4, process_index=0, coords=(0,2,0), core_on_chip=0)
I0329 09:51:37.176956      74 pjrt_client.cc:555] Addressable PjRt-IFRT device: TpuDevice(id=5, pr

Part 1
doing it using jit

In [2]:
x=2
y=4
mesh = jax.make_mesh(axis_shapes=(x, y), axis_names=('x', 'y'),axis_types=(shd.AxisType.Explicit, shd.AxisType.Explicit))
sharding = NamedSharding(mesh, P('x', 'y'))
jax.set_mesh(mesh)

print(jax.devices())

[TpuDevice(id=0, process_index=0, coords=(0,0,0), core_on_chip=0), TpuDevice(id=1, process_index=0, coords=(1,0,0), core_on_chip=0), TpuDevice(id=2, process_index=0, coords=(0,1,0), core_on_chip=0), TpuDevice(id=3, process_index=0, coords=(1,1,0), core_on_chip=0), TpuDevice(id=4, process_index=0, coords=(0,2,0), core_on_chip=0), TpuDevice(id=5, process_index=0, coords=(1,2,0), core_on_chip=0), TpuDevice(id=6, process_index=0, coords=(0,3,0), core_on_chip=0), TpuDevice(id=7, process_index=0, coords=(1,3,0), core_on_chip=0)]


In [3]:
p = 2
q = 2
array = jnp.arange(p * x * q * y, dtype=jnp.int32).reshape(( x,p,  y,q))
sharding = NamedSharding(mesh, P('x',None, 'y',None))
array = jax.device_put(array, sharding)

In [4]:
@jax.jit
def average(arr):
    print(jax.typeof(arr))
    arr=jnp.mean(arr,axis=(1,3))
    return jax.lax.with_sharding_constraint(arr, P('x', 'y'))
    

In [5]:
print(array.reshape((x * p, y * q), out_sharding=NamedSharding(mesh, P(None, None))))
print(average(array))

[[ 0  1  2  3  4  5  6  7]
 [ 8  9 10 11 12 13 14 15]
 [16 17 18 19 20 21 22 23]
 [24 25 26 27 28 29 30 31]]
int32[2@x,2,4@y,2]
[[ 4.5  6.5  8.5 10.5]
 [20.5 22.5 24.5 26.5]]


Using Shard map

In [6]:
@jax.shard_map(in_specs=jax.P('x', 'y'), out_specs=jax.P('x','y'))
def avg_shard_mad(arr):
    assert arr.shape == (p,q)
    return jnp.mean(arr).reshape((1,1))

In [7]:
p = 2
q = 2
array = jnp.arange(p * x * q * y, dtype=jnp.int32).reshape(( x*p,  y*q))
sharding = NamedSharding(mesh, P('x', 'y'))
array = jax.device_put(array, sharding)
print(array)
with jax.profiler.trace("/kaggle/working/shardmap_avg"):
    out=avg_shard_mad(array).reshape((x,y))
    _ = jax.block_until_ready(out)
print(out)

[[ 0  1  2  3  4  5  6  7]
 [ 8  9 10 11 12 13 14 15]
 [16 17 18 19 20 21 22 23]
 [24 25 26 27 28 29 30 31]]


I0329 09:51:37.643424      74 profiler_session.cc:103] Profiler session initializing.
I0329 09:51:37.643442      74 profiler_session.cc:118] Profiler session started.
I0329 09:51:37.983072      74 profiler_session.cc:68] Profiler session collecting data.


[[ 4.5  6.5  8.5 10.5]
 [20.5 22.5 24.5 26.5]]


I0329 09:51:38.304105      74 save_profile.cc:150] Collecting XSpace to repository: /kaggle/working/shardmap_avg/plugins/profile/2026_03_29_09_51_38/fe33117065c7.xplane.pb
I0329 09:51:38.338730      74 save_profile.cc:123] Creating directory: /kaggle/working/shardmap_avg/plugins/profile/2026_03_29_09_51_38

I0329 09:51:38.380667      74 save_profile.cc:129] Dumped gzipped tool data for trace.json.gz to /kaggle/working/shardmap_avg/plugins/profile/2026_03_29_09_51_38/fe33117065c7.trace.json.gz
I0329 09:51:38.384386      74 profiler_session.cc:136] Profiler session tear down.


Now the roll function implementation

In [8]:
@jax.shard_map(in_specs=(jax.P('x', 'y'),jax.P('x', 'y'),jax.P()), out_specs=jax.P('x','y'))
def roll_diff(base,roll,shift):
    return base-jnp.roll(roll,shift=shift,axis=0)

In [9]:
p = 2
q = 2
array = jnp.arange(p * x * q * y, dtype=jnp.int32).reshape(( x*p,  y*q))
sharding = NamedSharding(mesh, P('x', 'y'))
array1 = jax.device_put(array, sharding)
array2=jax.device_put(array, sharding)
print(array)
with jax.profiler.trace("/kaggle/working/roll_diff"):
    out=roll_diff(array1,array2,1)
    _ = jax.block_until_ready(out)
print(out)

[[ 0  1  2  3  4  5  6  7]
 [ 8  9 10 11 12 13 14 15]
 [16 17 18 19 20 21 22 23]
 [24 25 26 27 28 29 30 31]]


I0329 09:51:38.436133      74 profiler_session.cc:103] Profiler session initializing.
I0329 09:51:38.436148      74 profiler_session.cc:118] Profiler session started.


I0329 09:51:38.929233      74 profiler_session.cc:68] Profiler session collecting data.


[[-8 -8 -8 -8 -8 -8 -8 -8]
 [ 8  8  8  8  8  8  8  8]
 [-8 -8 -8 -8 -8 -8 -8 -8]
 [ 8  8  8  8  8  8  8  8]]


I0329 09:51:39.267505      74 save_profile.cc:150] Collecting XSpace to repository: /kaggle/working/roll_diff/plugins/profile/2026_03_29_09_51_39/fe33117065c7.xplane.pb
I0329 09:51:39.328525      74 save_profile.cc:123] Creating directory: /kaggle/working/roll_diff/plugins/profile/2026_03_29_09_51_39

I0329 09:51:39.400325      74 save_profile.cc:129] Dumped gzipped tool data for trace.json.gz to /kaggle/working/roll_diff/plugins/profile/2026_03_29_09_51_39/fe33117065c7.trace.json.gz
I0329 09:51:39.407549      74 profiler_session.cc:136] Profiler session tear down.


**Problem 2:** Here we’ll make a basic “mixture of experts” model together. Let **W**: float32[E_X, D, F] be a set of E “expert” matrices. Let **A**: float32[S_X, D] (our activations) and let **B**: int32[S_X] be a set of “routing assignments” where B[i] is an integer in the range `[0, E)` telling us which matrix we want to process that activation. We want to write a function in JAX that returns `Out[i] = W[B[i]] @ A[i]`.

1. Let’s start by ignoring sharding altogether. Make all of these tensors small enough so they fit in one device. Write a local implementation of this function. *Make sure you don’t materialize an array of shape `[S, D, F]`! Hint: try sorting the tokens into a new buffer of shape `[E, S, D]` with some attention to masking (why do we need the second dimension to have size S?).*

2. If you just `jax.jit` the above method, something will happen. Profile this and see what communication it decided to do. How long does it take?

3. One problem you’ll notice with the above is that it likely gathers the full set of activations **A** locally, i.e. AllGather_X([S_X, D]). Not only is this expensive communication-wise, it’s also incredibly expensive memory-wise if we can’t fit the full set of activations locally. Implement the above using `shard_map` and explicit communication.
    1. For a first pass, it might be easiest to use a `jax.lax.all_gather` and reorder as in (a).
    2. For a second pass, try to avoid materializing any array of size `[E, S, D]`, i.e. try to perform the computation in a ragged fashion using a `jax.lax.all_to_all` inside a `jax.lax.while_loop`. This way, you can avoid materializing the full activations and wasting compute on padding. How much faster is this than your original implementation?

4. Most MoEs route to multiple (k) experts and then average the result. Refactor the above to implement this. Let **B**: int32[S, k] in this case for the k experts to route to.

part 1

In [10]:
@jax.jit
def moe_local(W: jnp.ndarray, A: jnp.ndarray, B: jnp.ndarray) -> jnp.ndarray:
    """
    Local implementation of MoE routing. for each s there is one expert which we need to process with input
    W: [E, D, F] - Expert weights
    A: [S, D]    - Token activations
    B: [S]       - Routing assignments (values in [0, E))

    output - (S,F)
    """
    E, D, F = W.shape
    S = A.shape[0]
    output=jnp.zeros((S,F))
    # 1. Inspect your inputs at the start of the function
    jax.debug.inspect_array_sharding(A, callback=lambda s: print(f"A Input Sharding: {s}"))
    
    def process_exp(carry,e):
        output=carry
        mask=(B==e)[:,None] #shape [S,1] of 0,1
        jax.debug.inspect_array_sharding(W, callback=lambda s: print(f"W Input Sharding: {s}"))
        to_add=A@W[e]  #shape [S,F]
        current_e_proccess=to_add*mask
        output=output+current_e_proccess
        return output, None
    out,_=jax.lax.scan(process_exp,output,jnp.arange(E))
    return out

In [11]:
x=8
y=1
mesh = jax.make_mesh(axis_shapes=(x, y), axis_names=('x', 'y'),axis_types=(shd.AxisType.Auto, shd.AxisType.Auto))
sharding = NamedSharding(mesh, P('x', 'y'))
jax.set_mesh(mesh)

print(jax.devices())
# 2. Define Dimensions
if jax.devices()[0].platform == 'tpu':
    E, S, D, F = 8, 2048, 4096, 14336 # for real test
else:
    E, S, D, F = 8, 16, 16, 16 # for CPU debugging

# 3. Initialize Data
key = jax.random.PRNGKey(0)
k1, k2, k3 = jax.random.split(key, 3)

W_data = jax.random.normal(k1, (E, D, F))
A_data = jax.random.normal(k2, (S, D))
B_data = jax.random.randint(k3, (S,), 0, E)

# 4. Define Sharding Strategies
# Tokens (S) are split across 'x', Experts (E) are split across 'y'
W_sharding = NamedSharding(mesh, P('x', None, None)) # Shard experts
A_sharding = NamedSharding(mesh, P('x', None))       # Shard tokens
B_sharding = NamedSharding(mesh, P('x'))             # Shard assignments same as tokens

# Move data to TPU/GPU with explicit sharding
W = jax.device_put(W_data, W_sharding)
A = jax.device_put(A_data, A_sharding)
B = jax.device_put(B_data, B_sharding)

with jax.profiler.trace("/kaggle/working/MoE_auto"):
    result = moe_local(W, A, B)
    _ = jax.block_until_ready(result)

print(f"Input A Sharding: {A.sharding}")
print(f"Output Shape: {result.shape}")
print(f"Output Sharding: {result.sharding}") # Should be P('x', None)
print(f"output[:5]: {jax.device_get(result)[:5]}")

[TpuDevice(id=0, process_index=0, coords=(0,0,0), core_on_chip=0), TpuDevice(id=1, process_index=0, coords=(1,0,0), core_on_chip=0), TpuDevice(id=2, process_index=0, coords=(0,1,0), core_on_chip=0), TpuDevice(id=3, process_index=0, coords=(1,1,0), core_on_chip=0), TpuDevice(id=4, process_index=0, coords=(0,2,0), core_on_chip=0), TpuDevice(id=5, process_index=0, coords=(1,2,0), core_on_chip=0), TpuDevice(id=6, process_index=0, coords=(0,3,0), core_on_chip=0), TpuDevice(id=7, process_index=0, coords=(1,3,0), core_on_chip=0)]


I0329 09:51:46.024034      74 profiler_session.cc:103] Profiler session initializing.
I0329 09:51:46.024069      74 profiler_session.cc:118] Profiler session started.


A Input Sharding: NamedSharding(mesh=Mesh('x': 8, 'y': 1, axis_types=(Auto, Auto)), spec=PartitionSpec('x',), memory_kind=device)
W Input Sharding: NamedSharding(mesh=Mesh('x': 8, 'y': 1, axis_types=(Auto, Auto)), spec=PartitionSpec('x',), memory_kind=device)


I0329 09:51:46.849207      74 profiler_session.cc:68] Profiler session collecting data.


Input A Sharding: NamedSharding(mesh=Mesh('x': 8, 'y': 1, axis_types=(Auto, Auto)), spec=PartitionSpec('x', None), memory_kind=device)
Output Shape: (2048, 14336)
Output Sharding: NamedSharding(mesh=Mesh('x': 8, 'y': 1, axis_types=(Auto, Auto)), spec=PartitionSpec('x',), memory_kind=device)
output[:5]: [[  18.885544    11.383617     5.1512413 ...  -38.091457   -62.212044
    26.361519 ]
 [ -13.4859085  121.00219      1.4184179 ...  -84.74631     53.94409
   -38.728035 ]
 [ 108.03144   -103.83809    -88.69052   ...  -86.89644    -31.94041
   -36.217625 ]
 [ -29.474749   -50.55308    -53.90491   ...   60.18759    -96.97791
    26.56     ]
 [  26.717096   147.05705     45.852036  ...   29.595524   -73.83397
    24.027351 ]]


I0329 09:51:47.152838      74 save_profile.cc:150] Collecting XSpace to repository: /kaggle/working/MoE_auto/plugins/profile/2026_03_29_09_51_47/fe33117065c7.xplane.pb
I0329 09:51:47.183746      74 save_profile.cc:123] Creating directory: /kaggle/working/MoE_auto/plugins/profile/2026_03_29_09_51_47

I0329 09:51:47.225525      74 save_profile.cc:129] Dumped gzipped tool data for trace.json.gz to /kaggle/working/MoE_auto/plugins/profile/2026_03_29_09_51_47/fe33117065c7.trace.json.gz
I0329 09:51:47.229004      74 profiler_session.cc:136] Profiler session tear down.


in this case while loop runs for like 73 ms

Now we shall implement the above algorithm to reduce the communication bound using all-to-all to reduce memory footprint and eliminate expensive all-gather

## Optimized Distributed MoE Routing (Expert Parallelism)

**The Problem:** A naive MoE implementation computes every expert on every token and masks out the unused results. This requires `E * S_x` dense computations per device, which is massively wasteful in terms of compute and memory.

**The Solution:** Instead of broadcasting all tokens to all experts (All-Gather), we dynamically route tokens only to their assigned experts using targeted network swaps (All-to-All). To satisfy XLA's requirement for static array shapes, we process tokens in chunks using a fixed **Expert Capacity** (`expert_num_token_process_size`).

### Notation Setup
* **`N`**: Total number of devices/TPUs.
* **`E`**: Total number of experts globally.
* **`E_x`**: Experts stored locally per device (`E / N`).
* **`S_x`**: Number of tokens stored locally per device.
* **`C`**: `expert_num_token_process_size` (Max tokens an expert processes per communication chunk).
* **`D`**: Hidden dimension of the tokens.
* **`F`**: Output dimension of the experts.

### Algorithm (Device POV)

**Inputs:**
* `W_local`: `[E_x, D, F]` -> Local expert weights on this device.
* `A_local`: `[S_x, D]` -> Local token activations.
* `B_local`: `[S_x]` -> Target expert indices for local tokens.

**Outputs:**
* `Out_local`: `[S_x, F]` -> Processed tokens.

---

**Loop:** While there are unprocessed tokens in `A_local`, do:

1.  **Local Binning & Padding (Prepare for Dispatch)**
    * Iterate through the next batch of local tokens.
    * Group tokens by their assigned expert `B[i]`.
    * Take up to `C` tokens for each of the `E` experts.
    * If an expert has fewer than `C` tokens assigned, pad the remainder with dummy zero-tokens.
    * *Keep track of the original token indices so we know where to put them back later.*
    * **Current Array Shape:** `[E, C, D]`

2.  **Forward Dispatch (Network Swapping)**
    * Perform an `all_to_all` communication across the `N` devices.
    * Send the padded token chunks to the devices that actually hold those specific experts.
    * Receive the tokens that other devices are sending to *our* local experts.
    * **Received Array Shape:** `[E_x, C * N, D]` *(We hold `E_x` experts, and we receive `C` tokens from each of the `N` devices).*

3.  **Dense Local Compute (No Masking Needed!)**
    * Multiply our received, tightly-packed tokens by our local expert weights.
    * `Processed_Tokens = W_local @ Received_Array`
    * Because we only received tokens meant for our experts, 0% of this computation is thrown away (excluding the static padding).
    * **Computed Array Shape:** `[E_x, C * N, F]`

4.  **Backward Return (Network Swapping)**
    * Perform a second `all_to_all` communication.
    * Send the processed tokens back to the devices they originally came from.
    * **Received Array Shape:** `[E, C, F]`

5.  **Un-binning & Scatter (Reassembly)**
    * Using the original token indices we saved in Step 1, extract the processed tokens from the `[E, C, F]` grid.
    * Write them into the final `Out_local` array of shape `[S_x, F]`.
    * Discard the padding.

**End Loop**

---
*By tuning the capacity parameter `C`, we can strictly control the trade-off between wasteful padding computation (if `C` is too large) and network latency overhead (if `C` is too small and requires too many `while_loop` iterations).*

In [12]:
x=8
y=1
mesh = jax.make_mesh(axis_shapes=(x, y), axis_names=('x', 'y'),axis_types=(shd.AxisType.Explicit, shd.AxisType.Explicit))
sharding = NamedSharding(mesh, P('x', 'y'))
jax.set_mesh(mesh)
print(jax.devices())
def get_pipelined_moe_router(E: int, C: int, max_tokens_per_expert: int):
    # This MUST be a standard Python integer for the 'for' loop to unroll properly
    static_loop_steps = (max_tokens_per_expert + C - 1) // C
    @jax.shard_map(
        mesh=None, 
        in_specs=(P('x', None, None), P('x', None), P('x')),
        out_specs=P('x', None),
        check_vma=True
    )
    def moe_router_sharded(W_local, A_local, B_local):
        S_x, D = A_local.shape
        _, _, F = W_local.shape
        
        with jax.named_scope("MoE_PreProcessing"):
            sort_idx = jnp.argsort(B_local)
            unsort_idx = jnp.argsort(sort_idx) 
            sorted_A = A_local[sort_idx]
            counts = jnp.bincount(B_local, length=E) 
            starts = jnp.cumsum(counts) - counts     
        
        def fetch_chunk(chunk_idx):
            with jax.named_scope("Pipeline_Comm_Fetch"):
                queue_idx = chunk_idx * C + jnp.arange(C)[None, :]
                base_idx = starts[:, None] + queue_idx
                valid = queue_idx < counts[:, None]
                safe_idx = jnp.where(valid, base_idx, 0)
                send_A = sorted_A[safe_idx]
                send_A = jnp.where(valid[:, :, None], send_A, 0.0)
                # [E, C, D] -> [E_x, N_x * C, D]
                recv_A = jax.lax.all_to_all(send_A, axis_name='x', split_axis=0, concat_axis=1)
                return recv_A, valid, safe_idx
        def compute_and_scatter(recv_A, valid, safe_idx, carry_out):
            with jax.named_scope("Pipeline_Compute"):
                recv_Out = jnp.einsum('edf,ecd->ecf', W_local, recv_A)
                # [E_x, N_x * C, F] -> [E, C, F]
                recv_Back = jax.lax.all_to_all(recv_Out, axis_name='x', split_axis=1, concat_axis=0)
                valid_results = jnp.where(valid[:, :, None], recv_Back, 0.0)
                return carry_out.at[safe_idx].add(valid_results)
        # --- PHASE 1: PRE-FETCH ---
        with jax.named_scope("Phase1_PreFetch"):
            initial_out = jnp.zeros((S_x, F), dtype=A_local.dtype)
            initial_out = jax.lax.pvary(initial_out, 'x')
            
            # Initialize our pipeline state variables
            carry_out = initial_out
            recv_A_curr, valid_curr, safe_idx_curr = fetch_chunk(0)
        
        num_scan_steps = max(0, static_loop_steps - 1)
        
        # --- PHASE 2: UNROLLED PIPELINE (Python For-Loop) ---
        with jax.named_scope("Phase2_Pipeline_Unrolled"):
            for i in range(num_scan_steps):
                # Dynamically name each step so it looks beautiful in the Perfetto trace
                with jax.named_scope(f"Unrolled_Step_{i}"):
                    next_chunk_idx = i + 1
                    
                    # 1. Trace the math FIRST (keeps MXU busy)
                    carry_out = compute_and_scatter(recv_A_curr, valid_curr, safe_idx_curr, carry_out)
                    
                    # 2. Trace the network SECOND (fetches in background)
                    # We just overwrite the _curr variables directly for the next iteration!
                    recv_A_curr, valid_curr, safe_idx_curr = fetch_chunk(next_chunk_idx)
        
        # --- PHASE 3: DRAIN ---
        with jax.named_scope("Phase3_Drain"):
            # Because static_loop_steps is a standard Python int, we can use a standard Python if-statement
            if static_loop_steps > 0:
                carry_out = compute_and_scatter(recv_A_curr, valid_curr, safe_idx_curr, carry_out)
        
        # --- POST-PROCESSING ---
        with jax.named_scope("MoE_PostProcessing"):
            final_out = carry_out[unsort_idx]
            
        return final_out
    return moe_router_sharded


[TpuDevice(id=0, process_index=0, coords=(0,0,0), core_on_chip=0), TpuDevice(id=1, process_index=0, coords=(1,0,0), core_on_chip=0), TpuDevice(id=2, process_index=0, coords=(0,1,0), core_on_chip=0), TpuDevice(id=3, process_index=0, coords=(1,1,0), core_on_chip=0), TpuDevice(id=4, process_index=0, coords=(0,2,0), core_on_chip=0), TpuDevice(id=5, process_index=0, coords=(1,2,0), core_on_chip=0), TpuDevice(id=6, process_index=0, coords=(0,3,0), core_on_chip=0), TpuDevice(id=7, process_index=0, coords=(1,3,0), core_on_chip=0)]


In [13]:

# 2. Define Dimensions
if jax.devices()[0].platform == 'tpu':
    E, S, D, F = 8, 2048, 4096, 14336 # for real test
else:
    E, S, D, F = 8, 16, 16, 16 # for CPU debugging

# 3. Initialize Data
key = jax.random.PRNGKey(0)
k1, k2, k3 = jax.random.split(key, 3)

W_data = jax.random.normal(k1, (E, D, F))
A_data = jax.random.normal(k2, (S, D))
B_data = jax.random.randint(k3, (S,), 0, E)

# 4. Define Sharding Strategies
# Tokens (S) are split across 'x', Experts (E) are split across 'y'
W_sharding = NamedSharding(mesh, P('x', None, None)) # Shard experts
A_sharding = NamedSharding(mesh, P('x', None))       # Shard tokens
B_sharding = NamedSharding(mesh, P('x'))             # Shard assignments same as tokens

# Move data to TPU/GPU with explicit sharding
W = jax.device_put(W_data, W_sharding)
A = jax.device_put(A_data, A_sharding)
B = jax.device_put(B_data, B_sharding)

# 1. Instantiate the router function
C = 32
moe_router_sharded = get_pipelined_moe_router(E, C, max((S//(E*E))*2,1))

print("AOT Compiling...")

# 1. Wrap the shard_map function in jax.jit to expose the .lower() API
jitted_router = jax.jit(moe_router_sharded)

# 2. Now you can lower and compile it for your specific shapes/sharding
compiled_router = jitted_router.lower(W, A, B).compile()
print("Compilation finished.")

print("Starting profiler...")
with jax.profiler.trace("/kaggle/working/MoE_chunk_all_to_all"):
    # 3. Call the explicitly compiled object
    result = compiled_router(W, A, B)
    _ = jax.block_until_ready(result)
print("Trace saved.")

# --- POST-PROCESSING ---
print(f"Input A Sharding: {A.sharding}")
print(f"Output Shape: {result.shape}")
print(f"Output Sharding: {result.sharding}") # Should be P('x', None)
print(f"output[:5]: {jax.device_get(result)[:5]}")

AOT Compiling...


/tmp/ipykernel_74/2282081637.py:48: DeprecationWarning: jax.lax.pvary is deprecated. Use `jax.lax.pcast(..., to='varying')
  initial_out = jax.lax.pvary(initial_out, 'x')


Compilation finished.
Starting profiler...


I0329 09:51:55.333227      74 profiler_session.cc:103] Profiler session initializing.
I0329 09:51:55.333248      74 profiler_session.cc:118] Profiler session started.
I0329 09:51:55.505819      74 profiler_session.cc:68] Profiler session collecting data.


Trace saved.
Input A Sharding: NamedSharding(mesh=Mesh('x': 8, 'y': 1, axis_types=(Explicit, Explicit)), spec=PartitionSpec('x', None), memory_kind=device)
Output Shape: (2048, 14336)
Output Sharding: NamedSharding(mesh=Mesh('x': 8, 'y': 1, axis_types=(Explicit, Explicit)), spec=PartitionSpec('x', None), memory_kind=device)
output[:5]: [[  18.885544    11.383617     5.1512413 ...  -38.091457   -62.212044
    26.361519 ]
 [ -13.4859085  121.00219      1.4184179 ...  -84.74631     53.94409
   -38.728035 ]
 [ 108.03144   -103.83809    -88.69052   ...  -86.89644    -31.94041
   -36.217625 ]
 [ -29.474749   -50.55308    -53.90491   ...   60.18759    -96.97791
    26.56     ]
 [  26.717096   147.05705     45.852036  ...   29.595524   -73.83397
    24.027351 ]]


I0329 09:51:55.809947      74 save_profile.cc:150] Collecting XSpace to repository: /kaggle/working/MoE_chunk_all_to_all/plugins/profile/2026_03_29_09_51_55/fe33117065c7.xplane.pb
I0329 09:51:55.813742      74 save_profile.cc:123] Creating directory: /kaggle/working/MoE_chunk_all_to_all/plugins/profile/2026_03_29_09_51_55

I0329 09:51:55.818494      74 save_profile.cc:129] Dumped gzipped tool data for trace.json.gz to /kaggle/working/MoE_chunk_all_to_all/plugins/profile/2026_03_29_09_51_55/fe33117065c7.trace.json.gz
I0329 09:51:55.819383      74 profiler_session.cc:136] Profiler session tear down.


in this case while loop runs time drop to 1.7 ms

# Now we make MOE which selects k experts from E experts

In [14]:
def get_pipelined_moe_router_topk(E: int, C: int, max_tokens_per_expert: int):
    static_loop_steps = (max_tokens_per_expert + C - 1) // C

    @jax.shard_map(
        mesh=None, 
        # B_local is now 2D [S_x, k], so its spec becomes P('x', None)
        in_specs=(P('x', None, None), P('x', None), P('x', None)),
        out_specs=P('x', None),
        check_vma=True
    )
    def moe_router_sharded(W_local, A_local, B_local):
        S_x, D = A_local.shape
        _, k = B_local.shape
        _, _, F = W_local.shape
        
        with jax.named_scope("MoE_PreProcessing"):
            # --- 1. FLATTEN AND DUPLICATE ---
            # B_local: [S_x, k] -> [S_x * k]
            flat_B = B_local.reshape(-1)
            
            # A_local: [S_x, D] -> [S_x * k, D]
            # jnp.repeat safely duplicates token[0] 'k' times, then token[1] 'k' times...
            flat_A = jnp.repeat(A_local, k, axis=0)
            
            # --- 2. STANDARD ROUTING PREP ---
            sort_idx = jnp.argsort(flat_B)
            unsort_idx = jnp.argsort(sort_idx) 
            
            sorted_A = flat_A[sort_idx]
            counts = jnp.bincount(flat_B, length=E) 
            starts = jnp.cumsum(counts) - counts     
        
        def fetch_chunk(chunk_idx):
            with jax.named_scope("Pipeline_Comm_Fetch"):
                queue_idx = chunk_idx * C + jnp.arange(C)[None, :]
                base_idx = starts[:, None] + queue_idx
                valid = queue_idx < counts[:, None]
                safe_idx = jnp.where(valid, base_idx, 0)
                send_A = sorted_A[safe_idx]
                send_A = jnp.where(valid[:, :, None], send_A, 0.0)
                
                # [E, C, D] -> [E_x, N_x * C, D]
                recv_A = jax.lax.all_to_all(send_A, axis_name='x', split_axis=0, concat_axis=1)
                return recv_A, valid, safe_idx

        def compute_and_scatter(recv_A, valid, safe_idx, carry_out):
            with jax.named_scope("Pipeline_Compute"):
                recv_Out = jnp.einsum('edf,ecd->ecf', W_local, recv_A)
                
                # [E_x, N_x * C, F] -> [E, C, F]
                recv_Back = jax.lax.all_to_all(recv_Out, axis_name='x', split_axis=1, concat_axis=0)
                valid_results = jnp.where(valid[:, :, None], recv_Back, 0.0)
                return carry_out.at[safe_idx].add(valid_results)

        with jax.named_scope("Phase1_PreFetch"):
            # Output buffer must now be sized for S_x * k tokens!
            initial_out = jnp.zeros((S_x * k, F), dtype=A_local.dtype)
            initial_out = jax.lax.pvary(initial_out, 'x')
            recv_A_next, valid_next, safe_idx_next = fetch_chunk(0)
        
        def process_chunk_pipelined(carry, loop_idx):
            carry_out, recv_A_curr, valid_curr, safe_idx_curr = carry
            next_chunk_idx = loop_idx + 1
            
            # 1. Trace math first
            new_carry_out = compute_and_scatter(recv_A_curr, valid_curr, safe_idx_curr, carry_out)
            # 2. Trace network second
            recv_A_new, valid_new, safe_idx_new = fetch_chunk(next_chunk_idx)
            
            next_carry = (new_carry_out, recv_A_new, valid_new, safe_idx_new)
            return next_carry, None

        num_scan_steps = max(0, static_loop_steps - 1)
        init_carry = (initial_out, recv_A_next, valid_next, safe_idx_next)
        
        with jax.named_scope("Phase2_Pipeline_Scan"):
            final_carry, _ = jax.lax.scan(
                process_chunk_pipelined, 
                init_carry, 
                jnp.arange(num_scan_steps)
            )
        
        carry_out_final, recv_A_last, valid_last, safe_idx_last = final_carry
        
        with jax.named_scope("Phase3_Drain"):
            def drain(out):
                return compute_and_scatter(recv_A_last, valid_last, safe_idx_last, out)
            sorted_out = jax.lax.cond(static_loop_steps > 0, drain, lambda x: x, carry_out_final)
        
        with jax.named_scope("MoE_PostProcessing"):
            # 1. Unsort back to [S_x * k, F]
            final_out_flat = sorted_out[unsort_idx]
            
            # 2. Reshape to split the tokens and the experts [S_x, k, F]
            final_out_k = final_out_flat.reshape(S_x, k, F)
            
            # 3. Average across the 'k' dimension
            final_out = jnp.mean(final_out_k, axis=1)
            
        return final_out

    return moe_router_sharded

In [15]:
k = 2  # e.g., Top-2 routing

# 2. Define Dimensions
if jax.devices()[0].platform == 'tpu':
    E, S, D, F = 8, 2048, 4096, 14336 # for real test
else:
    E, S, D, F = 8, 16, 16, 16 # for CPU debugging

# 3. Initialize Data
key = jax.random.PRNGKey(0)
k1, k2, k3 = jax.random.split(key, 3)

W_data = jax.random.normal(k1, (E, D, F))
A_data = jax.random.normal(k2, (S, D))

# --- CHANGE 2: B_data is now a 2D matrix of shape (S, k) ---
B_data = jax.random.randint(k3, (S, k), 0, E)

# 4. Define Sharding Strategies
W_sharding = NamedSharding(mesh, P('x', None, None)) 
A_sharding = NamedSharding(mesh, P('x', None))       

# --- CHANGE 3: B_sharding must now accommodate the 2nd dimension ---
B_sharding = NamedSharding(mesh, P('x', None))              

# Move data to TPU/GPU with explicit sharding
W = jax.device_put(W_data, W_sharding)
A = jax.device_put(A_data, A_sharding)
B = jax.device_put(B_data, B_sharding)

# --- CHANGE 4: Scale the max capacity by 'k' and call the new function ---
C = 32
# Because each token is duplicated 'k' times, the maximum possible load on an expert increases!
max_capacity = max(((S * k) // (E * E)) * 2,1) 

moe_router_sharded = get_pipelined_moe_router_topk(E, C, max_capacity)

print("AOT Compiling...")

# 1. Wrap the shard_map function in jax.jit to expose the .lower() API
jitted_router = jax.jit(moe_router_sharded)

# 2. Now you can lower and compile it for your specific shapes/sharding
compiled_router = jitted_router.lower(W, A, B).compile()
print("Compilation finished.")

print("Starting profiler...")
# Updated the trace folder name so you don't overwrite your previous 1-expert traces
with jax.profiler.trace("/kaggle/working/MoE_chunk_all_to_all_topk"):
    # 3. Call the explicitly compiled object
    result = compiled_router(W, A, B)
    _ = jax.block_until_ready(result)
print("Trace saved.")

# --- POST-PROCESSING ---
print(f"Input A Sharding: {A.sharding}")
print(f"Output Shape: {result.shape}")
print(f"Output Sharding: {result.sharding}") # Should be P('x', None)
print(f"output[:5]: {jax.device_get(result)[:5]}")

AOT Compiling...


Compilation finished.
Starting profiler...


I0329 09:51:58.248858      74 profiler_session.cc:103] Profiler session initializing.
I0329 09:51:58.248888      74 profiler_session.cc:118] Profiler session started.
I0329 09:51:58.419941      74 profiler_session.cc:68] Profiler session collecting data.


Trace saved.
Input A Sharding: NamedSharding(mesh=Mesh('x': 8, 'y': 1, axis_types=(Explicit, Explicit)), spec=PartitionSpec('x', None), memory_kind=device)
Output Shape: (2048, 14336)
Output Sharding: NamedSharding(mesh=Mesh('x': 8, 'y': 1, axis_types=(Explicit, Explicit)), spec=PartitionSpec('x', None), memory_kind=device)
output[:5]: [[-6.4543791e+00 -6.1447357e+01  1.4763766e+01 ... -4.1468163e+00
  -4.5191868e+01 -9.9825306e+00]
 [ 4.2254906e+01 -2.9817375e+01 -8.9965363e+01 ... -4.0951706e+01
   5.8543129e+01 -3.3957718e+01]
 [ 2.5619011e+01  2.7922562e+01  3.2910114e+01 ...  9.6106758e+00
  -3.0259628e+01 -1.4785311e+01]
 [-8.0787949e+01 -7.9681587e+00  1.1352539e-02 ... -2.1164875e+00
  -6.0417374e+01  6.0471760e+01]
 [-4.3418076e+01  3.9231400e+00  6.0529003e+01 ...  2.8005562e+01
  -2.2898136e+01 -4.5419518e+01]]


I0329 09:51:58.718924      74 save_profile.cc:150] Collecting XSpace to repository: /kaggle/working/MoE_chunk_all_to_all_topk/plugins/profile/2026_03_29_09_51_58/fe33117065c7.xplane.pb
I0329 09:51:58.725137      74 save_profile.cc:123] Creating directory: /kaggle/working/MoE_chunk_all_to_all_topk/plugins/profile/2026_03_29_09_51_58

I0329 09:51:58.733538      74 save_profile.cc:129] Dumped gzipped tool data for trace.json.gz to /kaggle/working/MoE_chunk_all_to_all_topk/plugins/profile/2026_03_29_09_51_58/fe33117065c7.trace.json.gz
I0329 09:51:58.735063      74 profiler_session.cc:136] Profiler session tear down.


Problem 3: The collective matmul example above is actually super relevant for real LLMs. Let’s tweak the example to do the full Transformer stack.

As an exercise, let’s start by implementing an AllReduce collective matmul, i.e. A[BX, DY] *D W[DY, F] -> Out[BX, F]. Note that the output isn’t replicated. The naive algorithm is discussed above, basically just a local matmul followed by an AllReduce. Try to make a comms overlapped “collective” version of this operation. Hint: tile over the output dimension and feel free to use jax.lax.psum (aka AllReduce). Note: due to the way XLA handles this, it may not actually be faster than the baseline.

The complement to the AllReduce collective matmul above is a ReduceScatter collective matmul, as in Tmp[BX, FY] *F W2[FY, D] -> Out[BX, DY]. This occurs in the down-projection matrix in a Transformer. Implement a collective, overlapped version of this in JAX. Be careful about passing only the minimal amount of data you need. Hint: try permuting the result as you accumulate it.

Put these two together into an end-to-end Transformer block that performs In[BX, DY] *D Win[D, FY] *F Wout[FY, D] -> Out[BX, DY] with overlapped communication.3 How much faster is this than a jax.jit implementation?

# Basic transforment with jit implementation

In [16]:
x=4
y=2
mesh = jax.make_mesh(axis_shapes=(x, y), axis_names=('x', 'y'),axis_types=(shd.AxisType.Auto, shd.AxisType.Auto))
sharding = NamedSharding(mesh, P('x', 'y'))
jax.set_mesh(mesh)
print(jax.devices())
@jax.jit
def compute_mlp(In, Win, Wout):
    # Evaluates left-to-right: (BX, D) @ (D, FY) -> (BX, FY) @ (FY, D) -> (BX, D)
    # The '@' operator automatically acts as the "auto axis" for the BX batch dimensions.
    return jax.lax.with_sharding_constraint(In @ Win @ Wout,sharding)

[TpuDevice(id=0, process_index=0, coords=(0,0,0), core_on_chip=0), TpuDevice(id=1, process_index=0, coords=(1,0,0), core_on_chip=0), TpuDevice(id=2, process_index=0, coords=(0,1,0), core_on_chip=0), TpuDevice(id=3, process_index=0, coords=(1,1,0), core_on_chip=0), TpuDevice(id=4, process_index=0, coords=(0,2,0), core_on_chip=0), TpuDevice(id=5, process_index=0, coords=(1,2,0), core_on_chip=0), TpuDevice(id=6, process_index=0, coords=(0,3,0), core_on_chip=0), TpuDevice(id=7, process_index=0, coords=(1,3,0), core_on_chip=0)]


In [17]:
# --- 1. Define Llama 8B / Debug Dimensions ---
if jax.devices()[0].platform == 'tpu':
    BX = 4096    # Batch size * Sequence Length (e.g., 1 * 4096)
    D = 4096     # Llama 8B Hidden Dimension
    FY = 14336   # Llama 8B FFN Intermediate Dimension
else:
    BX = 16
    D = 32
    FY = 128

# --- 2. Initialize Data ---
key = jax.random.PRNGKey(0)
k1, k2, k3 = jax.random.split(key, 3)

In_data = jax.random.normal(k1, (BX, D))
Win_data = jax.random.normal(k2, (D, FY))
Wout_data = jax.random.normal(k3, (FY, D))

# --- 3. Define Standard 2D Sharding Strategies ---
# In: Shard the batch dimension across 'x' (Data Parallelism)
In_sharding = NamedSharding(mesh, P('x', 'y'))    

# Win: Shard the expanded feature dimension across 'y' (Tensor Parallelism - Column Linear)
Win_sharding = NamedSharding(mesh, P(None, 'y'))   

# Wout: Shard the contracted feature dimension across 'y' (Tensor Parallelism - Row Linear)
Wout_sharding = NamedSharding(mesh, P('y', None))  

# Move data to TPU/GPU
In = jax.device_put(In_data, In_sharding)
Win = jax.device_put(Win_data, Win_sharding)
Wout = jax.device_put(Wout_data, Wout_sharding)

# --- 4. AOT Compilation & Warmup ---
print("AOT Compiling...")
# Explicitly compile the function for the exact shapes and shardings of our inputs
compiled_mlp = compute_mlp.lower(In, Win, Wout).compile()
print("Compilation finished.")

# --- 5. Profile Execution ---
print("Starting profiler...")
with jax.profiler.trace("/kaggle/working/jit_MLP_Auto_Shard"):
    # Call the explicitly compiled object for the profiled run
    result = compiled_mlp(In, Win, Wout)
    _ = jax.block_until_ready(result)
print("Trace saved.")

# --- Post-Processing ---
print(f"Input Shape: {In.shape}, Sharding: {In.sharding}")
print(f"Output Shape: {result.shape}")
# Because 'In' was sharded on 'x', GSPMD will automatically ensure the output is also sharded on 'x'!
print(f"Output Sharding: {result.sharding}")

AOT Compiling...


Compilation finished.
Starting profiler...


I0329 09:52:03.317438      74 profiler_session.cc:103] Profiler session initializing.
I0329 09:52:03.317459      74 profiler_session.cc:118] Profiler session started.
I0329 09:52:03.497691      74 profiler_session.cc:68] Profiler session collecting data.


Trace saved.
Input Shape: (4096, 4096), Sharding: NamedSharding(mesh=Mesh('x': 4, 'y': 2, axis_types=(Auto, Auto)), spec=PartitionSpec('x', 'y'), memory_kind=device)
Output Shape: (4096, 4096)
Output Sharding: NamedSharding(mesh=Mesh('x': 4, 'y': 2, axis_types=(Auto, Auto)), spec=PartitionSpec('x', 'y'), memory_kind=device)


I0329 09:52:03.789878      74 save_profile.cc:150] Collecting XSpace to repository: /kaggle/working/jit_MLP_Auto_Shard/plugins/profile/2026_03_29_09_52_03/fe33117065c7.xplane.pb
I0329 09:52:03.791704      74 save_profile.cc:123] Creating directory: /kaggle/working/jit_MLP_Auto_Shard/plugins/profile/2026_03_29_09_52_03

I0329 09:52:03.793879      74 save_profile.cc:129] Dumped gzipped tool data for trace.json.gz to /kaggle/working/jit_MLP_Auto_Shard/plugins/profile/2026_03_29_09_52_03/fe33117065c7.trace.json.gz
I0329 09:52:03.794067      74 profiler_session.cc:136] Profiler session tear down.


This takes like 1.2ms with sharding constrain and 0.2ms is of Allreduce and 0.1 all gather
# Now we will implement manually with jax.lax.psum
it may nopt be actually faster because of XLA internal optimisation

In [18]:
x=4
y=2
mesh = jax.make_mesh(axis_shapes=(x, y), axis_names=('x', 'y'),axis_types=(shd.AxisType.Explicit, shd.AxisType.Explicit))
sharding = NamedSharding(mesh, P('x', 'y'))
jax.set_mesh(mesh)
@jax.jit
@jax.shard_map(
    mesh=mesh,
    in_specs=(P('x','y'),P('y',None)),
    out_specs=(P('x',None))
)
def compute_matmul_allreduce(A,W):
    # Evaluates left-to-right: A[BX, DY] *D W[DY, F] -> Out[BX, F]
    return jax.lax.psum(A@W,axis_name='y')
    # return jax.lax.with_sharding_constraint(A@W,NamedSharding(mesh, P('x', None)))

In [19]:
# --- 1. Define Llama 8B / Debug Dimensions ---
if jax.devices()[0].platform == 'tpu':
    BX = 4096    # Batch size * Sequence Length
    D = 4096     # Contraction Dimension (DY)
    F = 14336    # Output Dimension (F)
else:
    BX = 16
    D = 32
    F = 128

# --- 2. Initialize Data ---
key = jax.random.PRNGKey(0)
k1, k2 = jax.random.split(key, 2)

A_data = jax.random.normal(k1, (BX, D))
W_data = jax.random.normal(k2, (D, F))

# --- 3. Define Explicit Sharding Strategies ---
# A: Shard the batch dimension across 'x' and model dimension across 'y'
A_sharding = NamedSharding(mesh, P('x', 'y'))    

# W: Shard the model dimension across 'y' (Row Linear)
W_sharding = NamedSharding(mesh, P('y', None))   

# Move data to TPU/GPU with the specified shardings
A = jax.device_put(A_data, A_sharding)
W = jax.device_put(W_data, W_sharding)

# --- 4. AOT Compilation & Warmup ---
print("AOT Compiling...")
# Explicitly compile the function for the exact shapes and shardings of A and W
compiled_matmul = compute_matmul_allreduce.lower(A, W).compile()
print("Compilation finished.")

# --- 5. Profile Execution ---
print("Starting profiler...")
with jax.profiler.trace("/kaggle/working/experimental_al_reduce_matmul"):
    # Call the explicitly compiled object during the profiled window
    result = compiled_matmul(A, W)
    _ = jax.block_until_ready(result)
print("Trace saved.")

# --- Post-Processing ---
print(f"Input A Shape: {A.shape}, Sharding: {A.sharding}")
print(f"Input W Shape: {W.shape}, Sharding: {W.sharding}")
print(f"Output Shape: {result.shape}")
print(f"Output Sharding: {result.sharding}")

AOT Compiling...


Compilation finished.
Starting profiler...


I0329 09:52:07.464605      74 profiler_session.cc:103] Profiler session initializing.
I0329 09:52:07.464625      74 profiler_session.cc:118] Profiler session started.
I0329 09:52:07.619272      74 profiler_session.cc:68] Profiler session collecting data.


Trace saved.
Input A Shape: (4096, 4096), Sharding: NamedSharding(mesh=Mesh('x': 4, 'y': 2, axis_types=(Explicit, Explicit)), spec=PartitionSpec('x', 'y'), memory_kind=device)
Input W Shape: (4096, 14336), Sharding: NamedSharding(mesh=Mesh('x': 4, 'y': 2, axis_types=(Explicit, Explicit)), spec=PartitionSpec('y', None), memory_kind=device)
Output Shape: (4096, 14336)
Output Sharding: NamedSharding(mesh=Mesh('x': 4, 'y': 2, axis_types=(Explicit, Explicit)), spec=PartitionSpec('x', None), memory_kind=device)


I0329 09:52:07.914462      74 save_profile.cc:150] Collecting XSpace to repository: /kaggle/working/experimental_al_reduce_matmul/plugins/profile/2026_03_29_09_52_07/fe33117065c7.xplane.pb
I0329 09:52:07.915872      74 save_profile.cc:123] Creating directory: /kaggle/working/experimental_al_reduce_matmul/plugins/profile/2026_03_29_09_52_07

I0329 09:52:07.917509      74 save_profile.cc:129] Dumped gzipped tool data for trace.json.gz to /kaggle/working/experimental_al_reduce_matmul/plugins/profile/2026_03_29_09_52_07/fe33117065c7.trace.json.gz
I0329 09:52:07.917651      74 profiler_session.cc:136] Profiler session tear down.


In [20]:
# --- Mesh Setup ---
x, y = 2, 4
mesh = jax.make_mesh(
    (x, y),
    ('x', 'y'),
    axis_types=(shd.AxisType.Explicit, shd.AxisType.Explicit)
)

# 2. Set mesh BEFORE defining functions
jax.set_mesh(mesh)

print("Mesh:", mesh)
print("Abstract:", jax.sharding.get_abstract_mesh())


@jax.jit
@jax.shard_map(
    mesh=mesh,
    in_specs=(P('x','y'), P('y',None)),
    out_specs=P('x','y'), # Notice the output is now sharded on 'y' as well!
    check_vma=True
)
def compute_matmul_reduce_scatter_psum(A_local, W_local):
    # 1. Local Matmul
    # A_local: [BX, FY]
    # W_local: [FY, D]
    # partial_out: [BX, D] (Incomplete sums, but full D dimension)
    with jax.named_scope("Local_Matmul"):
        partial_out = A_local @ W_local
        
    # 2. Reduce-Scatter Network Communication
    # Sum the partial results across the 'y' axis, and simultaneously 
    # scatter (slice) the D dimension (axis 1) across the 'y' axis devices.
    # final_out: [BX, DY]
    with jax.named_scope("ReduceScatter"):
        final_out = jax.lax.psum_scatter(
            partial_out, 
            axis_name='y', 
            scatter_dimension=1, 
            tiled=True # Use tiled=True for clean chunking without padding
        )
        
    return final_out


@jax.jit
@jax.shard_map(
    mesh=mesh,
    in_specs=(P('x','y'), P('y',None)),
    out_specs=P('x','y'), 
    check_vma=True
)
def compute_matmul_optimized_reduce_scatter(A_local, W_local):
    with jax.named_scope("Local_Matmul"):
        # partial_out: [BX, D]
        state = A_local @ W_local
        
    with jax.named_scope("Manual_Ring_ReduceScatter"):
        BX, D = state.shape
        DY = D // y
        my_id = jax.lax.axis_index('y')
        send_right_perm = tuple((j, (j + 1) % y) for j in range(y))
        
        # --- Unrolled Ring Loop ---
        # By using a standard Python loop, XLA flattens the execution graph.
        # This allows the network transfer of Step 2 to overlap with the addition of Step 1!
        for step in range(y - 1):
            with jax.named_scope(f"Ring_Step_{step}"):
                
                # 1. Calculate which chunk index to send and receive
                send_chunk_idx = (my_id - step - 1) % y
                recv_chunk_idx = (my_id - step - 2) % y
                
                # Convert chunk index to actual column offsets
                send_start = send_chunk_idx * DY
                recv_start = recv_chunk_idx * DY
                
                # 2. Slice out the specific block to send [BX, DY]
                # jax.lax.dynamic_slice(array, start_indices, slice_sizes)
                send_data = jax.lax.dynamic_slice(
                    state, 
                    (0, send_start), # Start at row 0, column 'send_start'
                    (BX, DY)         # Grab a block of size BX rows by DY columns
                )
                
                # 3. Fire the network transfer
                recv_data = jax.lax.ppermute(
                    send_data, 
                    axis_name='y', 
                    perm=send_right_perm
                )
                
                # 4. In-place addition logic
                # Extract the target memory block
                target_slice = jax.lax.dynamic_slice(state, (0, recv_start), (BX, DY))
                # Add the received network data
                updated_slice = target_slice + recv_data
                # Slam it back into the main matrix (XLA does this in-place)
                state = jax.lax.dynamic_update_slice(
                    state, 
                    updated_slice, 
                    (0, recv_start)
                )
                
        # 5. Extract the final fully summed chunk for this specific device
        final_out = jax.lax.dynamic_slice(
            state, 
            (0, my_id * DY), 
            (BX, DY)
        )
        
    return final_out

Mesh: Mesh('x': 2, 'y': 4, axis_types=(Explicit, Explicit))
Abstract: AbstractMesh('x': 2, 'y': 4, axis_types=(Explicit, Explicit), device_kind=TPU v5 lite, num_cores=1)


In [21]:
# --- 1. Define Llama 8B / Debug Dimensions ---
if jax.devices()[0].platform == 'tpu':
    BX = 4096    # Batch size * Sequence Length
    F = 14336    # Contraction Dimension (FY)
    D = 4096     # Output Dimension (DY)
else:
    BX = 16
    F = 128
    D = 32

# --- 2. Initialize Data ---
key = jax.random.PRNGKey(0)
k1, k2 = jax.random.split(key, 2)

A_data = jax.random.normal(k1, (BX, F))
W_data = jax.random.normal(k2, (F, D))

A_sharding = NamedSharding(mesh, P('x', 'y'))    
W_sharding = NamedSharding(mesh, P('y', None))   

A = jax.device_put(A_data, A_sharding)
W = jax.device_put(W_data, W_sharding)

# --- 4. AOT Compilation & Warmup ---
print("AOT Compiling Manual Ring...")
compiled_manual_rs = compute_matmul_optimized_reduce_scatter.lower(A, W).compile()
print("Compilation finished.")

print("AOT Compiling Manual Ring...")
compiled_psum_rs = compute_matmul_reduce_scatter_psum.lower(A, W).compile()
print("Compilation finished.")

# --- 5. Profile Execution ---
print("Starting profiler...")
with jax.profiler.trace("/kaggle/working/experimental_reduce_scatter_matmul"):
    # Call the explicitly compiled object during the profiled window
    result = compiled_manual_rs(A, W)
    _ = jax.block_until_ready(result)
print("Trace saved.")

# --- Post-Processing ---
print(f"result Shape (Global): {result.shape}")
print(f"result Sharding: {result.sharding}")
print(f"result[:5,:]=  {jax.device_get(result)[:5]}")

print("Starting profiler...")
with jax.profiler.trace("/kaggle/working/experimental_reduce_scatter_matmul_psum"):
    # Call the explicitly compiled object during the profiled window
    result = compiled_psum_rs(A, W)
    _ = jax.block_until_ready(result)
print("Trace saved.")


# --- Post-Processing ---
print(f"result Shape (Global): {result.shape}")
print(f"result Sharding: {result.sharding}")
print(f"result[:5,:]=  {jax.device_get(result)[:5]}")

AOT Compiling Manual Ring...


Compilation finished.
AOT Compiling Manual Ring...


Compilation finished.
Starting profiler...


I0329 09:52:12.490287      74 profiler_session.cc:103] Profiler session initializing.
I0329 09:52:12.490307      74 profiler_session.cc:118] Profiler session started.
I0329 09:52:12.671710      74 profiler_session.cc:68] Profiler session collecting data.


I0329 09:52:12.976591      74 save_profile.cc:150] Collecting XSpace to repository: /kaggle/working/experimental_reduce_scatter_matmul/plugins/profile/2026_03_29_09_52_12/fe33117065c7.xplane.pb
I0329 09:52:12.978806      74 save_profile.cc:123] Creating directory: /kaggle/working/experimental_reduce_scatter_matmul/plugins/profile/2026_03_29_09_52_12

I0329 09:52:12.981223      74 save_profile.cc:129] Dumped gzipped tool data for trace.json.gz to /kaggle/working/experimental_reduce_scatter_matmul/plugins/profile/2026_03_29_09_52_12/fe33117065c7.trace.json.gz
I0329 09:52:12.981503      74 profiler_session.cc:136] Profiler session tear down.
I0329 09:52:12.998824      74 profiler_session.cc:103] Profiler session initializing.
I0329 09:52:12.998863      74 profiler_session.cc:118] Profiler session started.
I0329 09:52:13.163745      74 profiler_session.cc:68] Profiler session collecting data.


Trace saved.
result Shape (Global): (4096, 4096)
result Sharding: NamedSharding(mesh=Mesh('x': 2, 'y': 4, axis_types=(Explicit, Explicit)), spec=PartitionSpec('x', 'y'), memory_kind=device)
result[:5,:]=  [[ 327.85727    -32.41047    -30.712112  ...  161.13254    -87.483765
  -254.78954  ]
 [ -89.57491   -109.89865    117.94324   ...  -52.747944    58.106747
     4.0105667]
 [ -79.301926   -72.62303     15.802834  ...   23.664978   -58.784973
   -18.380342 ]
 [ -81.36865   -185.74174    155.59047   ...   38.542217  -114.15207
    21.824621 ]
 [-120.00119   -182.76666    -88.433365  ...   14.850555   135.06818
  -116.05162  ]]
Starting profiler...


Trace saved.
result Shape (Global): (4096, 4096)
result Sharding: NamedSharding(mesh=Mesh('x': 2, 'y': 4, axis_types=(Explicit, Explicit)), spec=PartitionSpec('x', 'y'), memory_kind=device)
result[:5,:]=  [[ 327.85724    -32.41047    -30.712112  ...  161.13254    -87.48376
  -254.78952  ]
 [ -89.57492   -109.89864    117.94323   ...  -52.747944    58.10676
     4.0105667]
 [ -79.30193    -72.623024    15.802834  ...   23.664982   -58.784985
   -18.38034  ]
 [ -81.36865   -185.74176    155.59047   ...   38.542217  -114.15207
    21.824621 ]
 [-120.00119   -182.76666    -88.43336   ...   14.850557   135.06818
  -116.051636 ]]


I0329 09:52:13.458421      74 save_profile.cc:150] Collecting XSpace to repository: /kaggle/working/experimental_reduce_scatter_matmul_psum/plugins/profile/2026_03_29_09_52_13/fe33117065c7.xplane.pb
I0329 09:52:13.459535      74 save_profile.cc:123] Creating directory: /kaggle/working/experimental_reduce_scatter_matmul_psum/plugins/profile/2026_03_29_09_52_13

I0329 09:52:13.460784      74 save_profile.cc:129] Dumped gzipped tool data for trace.json.gz to /kaggle/working/experimental_reduce_scatter_matmul_psum/plugins/profile/2026_03_29_09_52_13/fe33117065c7.trace.json.gz
I0329 09:52:13.460905      74 profiler_session.cc:136] Profiler session tear down.


the original psum reduce scatter of jax give total latency of 1.01ms seconds and the single direction communication gives latenty of 1.2ms but we can improve it using bidirection communication as below

# bidirection communication for reduce scatter

In [22]:
x = 2
y = 4
mesh = jax.make_mesh(
    axis_shapes=(x, y), 
    axis_names=('x', 'y'),
    axis_types=(shd.AxisType.Explicit, shd.AxisType.Explicit)
)
jax.set_mesh(mesh)
@jax.jit
@jax.shard_map(
    mesh=mesh,
    in_specs=(P('x','y'), P('y',None)),
    out_specs=P('x','y'), 
    check_vma=True
)
def compute_matmul_recursive_halving_rs(A_local, W_local):
    with jax.named_scope("Local_Matmul"):
        # Initial state: [BX, D]
        state = A_local @ W_local
        
    with jax.named_scope("Recursive_Halving_ReduceScatter"):
        BX = state.shape[0]
        my_id = jax.lax.axis_index('y')
        
        # Calculate log2(y) to get the exact number of jumps needed
        num_steps = int(math.log2(y))
        
        # We loop backwards: e.g., for y=8, steps are 2, 1, 0
        for step in reversed(range(num_steps)):
            with jax.named_scope(f"Halving_Jump_{step}"):
                
                # The size of the array cuts in half at every single jump!
                current_size = state.shape[1]
                half_size = current_size // 2
                
                # MAGIC MATH: The k-th bit of our device ID tells us exactly 
                # which half of the array we need to keep, and which to send.
                # If bit is 0: keep first half. If bit is 1: keep second half.
                bit_is_one = (my_id >> step) & 1
                
                # Calculate dynamic slice offsets
                keep_offset = bit_is_one * half_size
                send_offset = (1 - bit_is_one) * half_size
                
                # 1. Slice out the half we don't need, to send to our partner
                send_data = jax.lax.dynamic_slice(
                    state, 
                    (0, send_offset), 
                    (BX, half_size)
                )
                # 2. Define the static 2-sided pairs for this jump using XOR (^)
                # Step 2: diff of 4. Step 1: diff of 2. Step 0: diff of 1.
                perm = tuple((j, j ^ (1 << step)) for j in range(y))
                
                # 3. Fire the 2-sided bidirectional exchange
                recv_data = jax.lax.ppermute(
                    send_data, 
                    axis_name='y', 
                    perm=perm
                )
                
                # 4. Slice out the half we ARE keeping
                keep_data = jax.lax.dynamic_slice(
                    state, 
                    (0, keep_offset), 
                    (BX, half_size)
                )
                
                # 5. Add them together! 
                # This inherently overwrites `state` with an array exactly half the size!
                state = keep_data + recv_data
                
        # By the end of the loop, `state` has shrunk perfectly from [BX, D] to [BX, DY]
        # and contains exactly the fully summed chunk belonging to this device!
        
    return state

In [23]:
# --- 1. Define Llama 8B / Debug Dimensions ---
if jax.devices()[0].platform == 'tpu':
    BX = 4096    # Batch size * Sequence Length
    F = 14336    # Contraction Dimension (FY)
    D = 4096     # Output Dimension (DY)
else:
    BX = 16
    F = 128
    D = 32

# --- 2. Initialize Data ---
key = jax.random.PRNGKey(0)
k1, k2 = jax.random.split(key, 2)

A_data = jax.random.normal(k1, (BX, F))
W_data = jax.random.normal(k2, (F, D))

A_sharding = NamedSharding(mesh, P('x', 'y'))    
W_sharding = NamedSharding(mesh, P('y', None))   

A = jax.device_put(A_data, A_sharding)
W = jax.device_put(W_data, W_sharding)

# --- 4. AOT Compilation & Warmup ---
print("AOT Compiling Recursive Halving Ring...")
compiled_halving_rs = compute_matmul_recursive_halving_rs.lower(A, W).compile()
print("Compilation finished.")


# --- 5. Profile Execution ---
print("Starting profiler...")
with jax.profiler.trace("/kaggle/working/experimental_recursive_halving_rs"):
    # Call the explicitly compiled object during the profiled window
    result = compiled_halving_rs(A, W)
    _ = jax.block_until_ready(result)
print("Trace saved.")

# --- Post-Processing ---
print(f"Result Shape (Global): {result.shape}")
print(f"Result Sharding: {result.sharding}")

# FIX: device_get(result) FIRST, then slice the resulting NumPy array SECOND
numpy_result = jax.device_get(result)
print(f"Result [first 5 rows]:\n{numpy_result[:5, :]}")

AOT Compiling Recursive Halving Ring...


Compilation finished.
Starting profiler...


I0329 09:52:14.238747      74 profiler_session.cc:103] Profiler session initializing.
I0329 09:52:14.238767      74 profiler_session.cc:118] Profiler session started.
I0329 09:52:14.409641      74 profiler_session.cc:68] Profiler session collecting data.


Trace saved.
Result Shape (Global): (4096, 4096)
Result Sharding: NamedSharding(mesh=Mesh('x': 2, 'y': 4, axis_types=(Explicit, Explicit)), spec=PartitionSpec('x', 'y'), memory_kind=device)
Result [first 5 rows]:
[[ 327.85727    -32.41047    -30.712116  ...  161.13254    -87.483765
  -254.78952  ]
 [ -89.57492   -109.89865    117.94324   ...  -52.747944    58.10676
     4.0105705]
 [ -79.30193    -72.623024    15.802834  ...   23.664982   -58.78498
   -18.380342 ]
 [ -81.36865   -185.74174    155.59047   ...   38.542217  -114.15206
    21.824621 ]
 [-120.00119   -182.76666    -88.43336   ...   14.850555   135.06819
  -116.05163  ]]


I0329 09:52:14.714890      74 save_profile.cc:150] Collecting XSpace to repository: /kaggle/working/experimental_recursive_halving_rs/plugins/profile/2026_03_29_09_52_14/fe33117065c7.xplane.pb
I0329 09:52:14.716918      74 save_profile.cc:123] Creating directory: /kaggle/working/experimental_recursive_halving_rs/plugins/profile/2026_03_29_09_52_14

I0329 09:52:14.719250      74 save_profile.cc:129] Dumped gzipped tool data for trace.json.gz to /kaggle/working/experimental_recursive_halving_rs/plugins/profile/2026_03_29_09_52_14/fe33117065c7.trace.json.gz
I0329 09:52:14.719501      74 profiler_session.cc:136] Profiler session tear down.


bidirecion communication gives latency of 1.1 micro seconds which is close to the original reduce scatter

# Final MLP of transformer

In [24]:
# --- 1. Mesh Setup (4x2) ---
x, y = 4, 2
mesh = jax.make_mesh(
    (x, y), 
    ('x', 'y'), 
    axis_types=(shd.AxisType.Explicit, shd.AxisType.Explicit)
)
jax.set_mesh(mesh)
def compute_matmul_optimized_reduce_scatter(A_local, W_local):
    with jax.named_scope("Local_Matmul"):
        # partial_out: [BX, D]
        state = A_local @ W_local
        
    with jax.named_scope("Manual_Ring_ReduceScatter"):
        BX, D = state.shape
        DY = D // y
        my_id = jax.lax.axis_index('y')
        send_right_perm = tuple((j, (j + 1) % y) for j in range(y))
        
        # --- Unrolled Ring Loop ---
        # By using a standard Python loop, XLA flattens the execution graph.
        # This allows the network transfer of Step 2 to overlap with the addition of Step 1!
        for step in range(y - 1):
            with jax.named_scope(f"Ring_Step_{step}"):
                
                # 1. Calculate which chunk index to send and receive
                send_chunk_idx = (my_id - step - 1) % y
                recv_chunk_idx = (my_id - step - 2) % y
                
                # Convert chunk index to actual column offsets
                send_start = send_chunk_idx * DY
                recv_start = recv_chunk_idx * DY
                
                # 2. Slice out the specific block to send [BX, DY]
                # jax.lax.dynamic_slice(array, start_indices, slice_sizes)
                send_data = jax.lax.dynamic_slice(
                    state, 
                    (0, send_start), # Start at row 0, column 'send_start'
                    (BX, DY)         # Grab a block of size BX rows by DY columns
                )
                
                # 3. Fire the network transfer
                recv_data = jax.lax.ppermute(
                    send_data, 
                    axis_name='y', 
                    perm=send_right_perm
                )
                
                # 4. In-place addition logic
                # Extract the target memory block
                target_slice = jax.lax.dynamic_slice(state, (0, recv_start), (BX, DY))
                # Add the received network data
                updated_slice = target_slice + recv_data
                # Slam it back into the main matrix (XLA does this in-place)
                state = jax.lax.dynamic_update_slice(
                    state, 
                    updated_slice, 
                    (0, recv_start)
                )
                
        # 5. Extract the final fully summed chunk for this specific device
        final_out = jax.lax.dynamic_slice(
            state, 
            (0, my_id * DY), 
            (BX, DY)
        )
        
    return final_out

# --- 2. Pure Helper Functions (No Decorators!) ---
def compute_matmul_recursive_halving_rs(A_local, W_local):
    """Custom Down-Projection: Matmul + Recursive Halving Reduce-Scatter"""
    with jax.named_scope("Local_Matmul"):
        # Initial state: [BX, D]
        state = A_local @ W_local
        
    with jax.named_scope("Recursive_Halving_ReduceScatter"):
        BX = state.shape[0]
        my_id = jax.lax.axis_index('y')
        
        num_steps = int(math.log2(y))
        
        for step in reversed(range(num_steps)):
            with jax.named_scope(f"Halving_Jump_{step}"):
                current_size = state.shape[1]
                half_size = current_size // 2
                
                bit_is_one = (my_id >> step) & 1
                keep_offset = bit_is_one * half_size
                send_offset = (1 - bit_is_one) * half_size
                
                send_data = jax.lax.dynamic_slice(state, (0, send_offset), (BX, half_size))
                perm = tuple((j, j ^ (1 << step)) for j in range(y))
                
                recv_data = jax.lax.ppermute(send_data, axis_name='y', perm=perm)
                keep_data = jax.lax.dynamic_slice(state, (0, keep_offset), (BX, half_size))
                
                state = keep_data + recv_data
                
    return state
    
def compute_overlapped_transformer_mlp(In_local, Win_local, Wout_local):
    """
    Overlapped Sequence Parallel MLP
    In: [BX, DY], Win: [D, FY_local], Wout: [FY_local, D]
    Returns: [BX, DY]
    """
    # --- LAYER 1: Up-Projection with All-Gather ---
    with jax.named_scope("UpProjection_AllGather"):
        # Gather 'DY' into 'D' across 'y'
        In_gathered = jax.lax.all_gather(In_local, axis_name='y', tiled=True, axis=1)
        
        # Local Matmul
        act_local = In_gathered @ Win_local
        act_local = jax.nn.gelu(act_local)

    # --- LAYER 2: Down-Projection with Custom Reduce-Scatter ---
    with jax.named_scope("DownProjection_Custom_ReduceScatter"):
        # Call our custom pure function directly! 
        # It handles both the matmul and the network communication.
        final_out = compute_matmul_recursive_halving_rs(act_local, Wout_local)
        # final_out = compute_matmul_optimized_reduce_scatter(act_local, Wout_local)
        
    return final_out

In [25]:
# --- 2. Define Dimensions & Initialize ---
if jax.devices()[0].platform == 'tpu':
    BX, D, FY = 4096, 4096, 14336
else:
    BX, D, FY = 16, 32, 128

key = jax.random.PRNGKey(0)
k1, k2, k3 = jax.random.split(key, 3)

# Sharding Definitions
# Input: sharded Batch ('x') and Hidden ('y')
in_sharding = NamedSharding(mesh, P('x', 'y'))
# Win: Column Parallel (FY sharded on 'y')
win_sharding = NamedSharding(mesh, P(None, 'y'))
# Wout: Row Parallel (FY sharded on 'y')
wout_sharding = NamedSharding(mesh, P('y', None))

In = jax.device_put(jax.random.normal(k1, (BX, D)), in_sharding)
Win = jax.device_put(jax.random.normal(k2, (D, FY)), win_sharding)
Wout = jax.device_put(jax.random.normal(k3, (FY, D)), wout_sharding)

shard_map_wrapper = jax.shard_map(
    mesh=mesh, 
    in_specs=(P('x', 'y'), P(None, 'y'), P('y', None)), 
    out_specs=P('x', 'y'), 
    check_vma=True
)

# 2. Pass your raw Python function into the wrapper
sharded_mlp = shard_map_wrapper(compute_overlapped_transformer_mlp)

# 3. Wrap that result in jax.jit
jitted_mlp = jax.jit(sharded_mlp)
# --- 3. AOT & Trace ---
print("AOT Compiling Overlapped MLP...")
compiled_mlp = jitted_mlp.lower(In, Win, Wout).compile()

print("Starting Profiler...")
with jax.profiler.trace("/kaggle/working/transformer_mlp_overlapped"):
    result = compiled_mlp(In, Win, Wout)
    _ = jax.block_until_ready(result)
print("Trace saved.")

# --- 4. Post-Processing ---
print(f"Result Shape: {result.shape}")
print(f"Result Sharding: {result.sharding}")
# Use the safe "Pull then Slice" method for printing
numpy_res = jax.device_get(result)
print(f"Result[:5, :5]:\n{numpy_res[:5, :5]}")

AOT Compiling Overlapped MLP...


Starting Profiler...


I0329 09:52:17.242273      74 profiler_session.cc:103] Profiler session initializing.
I0329 09:52:17.242300      74 profiler_session.cc:118] Profiler session started.
I0329 09:52:17.472713      74 profiler_session.cc:68] Profiler session collecting data.


Trace saved.
Result Shape: (4096, 4096)
Result Sharding: NamedSharding(mesh=Mesh('x': 4, 'y': 2, axis_types=(Explicit, Explicit)), spec=PartitionSpec('x', 'y'), memory_kind=device)
Result[:5, :5]:
[[ 3.0018091e+02 -7.1581860e+03  3.1050869e+03  7.6389258e+02
   1.3106394e+03]
 [ 1.9517635e+03 -9.7964807e+02  1.1894086e+04  5.2273926e+03
   5.0399556e+03]
 [ 2.7942241e+03  9.4629639e+02 -2.8077004e+03 -3.4709849e+03
  -1.1596265e+02]
 [ 7.0421958e+03 -1.9778638e+03  4.1506348e+00  2.7992432e+03
   1.9317000e+03]
 [ 4.4447695e+03  5.0696758e+03 -3.2665806e+03 -4.4762344e+03
  -1.2131782e+03]]


I0329 09:52:17.773356      74 save_profile.cc:150] Collecting XSpace to repository: /kaggle/working/transformer_mlp_overlapped/plugins/profile/2026_03_29_09_52_17/fe33117065c7.xplane.pb
I0329 09:52:17.775417      74 save_profile.cc:123] Creating directory: /kaggle/working/transformer_mlp_overlapped/plugins/profile/2026_03_29_09_52_17

I0329 09:52:17.777588      74 save_profile.cc:129] Dumped gzipped tool data for trace.json.gz to /kaggle/working/transformer_mlp_overlapped/plugins/profile/2026_03_29_09_52_17/fe33117065c7.trace.json.gz
I0329 09:52:17.777915      74 profiler_session.cc:136] Profiler session tear down.


it takes about 1.25ms similer to as jit implementation